# TiA 4 — Self-supervised representation learning

**Big question:** Can a useful representation emerge without class labels?

This intentionally tiny contrastive experiment tests the effect of augmentation assumptions. Labels are used only after representation learning, in a linear probe.

## How to work through this activity

This is a guided investigation rather than a coding tutorial. For each experiment:

1. Read the mathematical claim and identify the quantity being measured.
2. Predict the qualitative result before running the code.
3. Run one cell at a time and inspect both values and plots.
4. Change only the suggested variable; rerun and explain what changed.
5. Answer the **Explain** questions in your own words.

The code contains more comments than production software intentionally. You are not expected to memorise framework syntax. Focus on the relationship between assumptions, measurements and conclusions.

## Notation and prediction

Two stochastic views $x_i^{(1)}=T_1(x_i)$ and $x_i^{(2)}=T_2(x_i)$ form a positive pair. With normalised representations $z_i=f_\theta(x_i)$, the InfoNCE loss for one direction is

$$\ell_i=-\log\frac{\exp(z_i^{(1)\top}z_i^{(2)}/\tau)}{\sum_j\exp(z_i^{(1)\top}z_j^{(2)}/\tau)},$$

where $\tau$ is temperature. The augmentation distribution defines which information should become invariant. Predict what happens if a positive pair can receive transformations that alter its semantic class.

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import torch

# Fix every random-number generator so that your plots match the reference run.
# After completing the guided activity, change the seed to test robustness.
SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)

# The default path is designed for a CPU. Set this to False only after the
# notebook works and you want to run longer variants.
FAST_MODE = True

def make_shape_images(n=600, size=12, centred=False, seed=7):
    """Create noisy horizontal/vertical bars without downloading data."""
    local_rng = np.random.default_rng(seed)
    images = local_rng.normal(0, 0.12, (n, 1, size, size)).astype("float32")
    labels = local_rng.integers(0, 2, n)
    positions = np.full(n, size // 2) if centred else local_rng.integers(2, size - 2, n)
    for image, label, position in zip(images, labels, positions):
        image[0, position, 2:-2] += label == 0
        image[0, 2:-2, position] += label == 1
    return images, labels.astype("int64")

from torch import nn
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA

In [ ]:
X,y=make_shape_images(800,seed=20); x=torch.tensor(X)
class Encoder(nn.Module):
    def __init__(self): super().__init__(); self.net=nn.Sequential(nn.Flatten(),nn.Linear(144,48),nn.ReLU(),nn.Linear(48,16))
    def forward(self,z): return nn.functional.normalize(self.net(z),dim=1)
def augment(z,bad=False):
    # Translation and noise preserve bar orientation, hence the class label.
    out=torch.roll(z,int(rng.integers(-2,3)),dims=-1)+.08*torch.randn_like(z)
    # In the bad condition, each view is independently transposed with 50%
    # probability. A positive pair can therefore disagree in orientation.
    return out.transpose(-1,-2) if bad and rng.random()<.5 else out
def contrastive_train(bad=False):
    enc=Encoder(); opt=torch.optim.Adam(enc.parameters(),lr=2e-3); epochs=12 if FAST_MODE else 30
    for _ in range(epochs):
        # The two calls generate independent views of the same source examples.
        ids=torch.randperm(len(x))[:256]; a,b=enc(augment(x[ids],bad)),enc(augment(x[ids],bad))
        # Every row's matching column is its positive; other columns are negatives.
        logits=a@b.T/.15; target=torch.arange(len(ids)); loss=(nn.functional.cross_entropy(logits,target)+nn.functional.cross_entropy(logits.T,target))/2
        opt.zero_grad();loss.backward();opt.step()
    return enc
def probe(features,n_labels=80):
    idx=np.r_[np.where(y==0)[0][:n_labels//2],np.where(y==1)[0][:n_labels//2]]; test=np.setdiff1d(np.arange(len(y)),idx)
    clf=LogisticRegression().fit(features[idx],y[idx]); return clf.score(features[test],y[test])
raw=X.reshape(len(X),-1); random_features=Encoder()(x).detach().numpy(); good=contrastive_train()(x).detach().numpy(); bad=contrastive_train(True)(x).detach().numpy()
probe_scores={name:probe(z) for name,z in [("pixels",raw),("random",random_features),("contrastive",good),("bad augmentation",bad)]}
for name,value in probe_scores.items(): print(f"{name:16s} linear-probe accuracy={value:.3f}")
assert probe_scores["contrastive"] > probe_scores["random"]+.20
assert probe_scores["bad augmentation"] < probe_scores["contrastive"]-.10

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(9,3))
for a,(name,z) in zip(ax,[("random encoder",random_features),("contrastive encoder",good)]):
    p=PCA(2).fit_transform(z); a.scatter(*p.T,c=y,s=8,cmap="coolwarm"); a.set_title(name)
plt.show()
similarity=good@good.T; np.fill_diagonal(similarity,-np.inf)
print("Nearest-neighbour label agreement:",np.mean(y[similarity.argmax(1)]==y))

### Explain

1. What invariance does the augmentation define?
2. Why is a linear probe evidence about representation geometry, not end-to-end performance?
3. Explain the bad-augmentation result without saying the optimiser failed.

**Reading:** [Chen et al., SimCLR](https://proceedings.mlr.press/v119/chen20j.html).

## Expected pattern and limits

The good contrastive encoder should improve linear-probe accuracy and nearest-neighbour agreement over a random encoder. A transformation that sometimes changes bar orientation should damage the class structure. This toy instance-discrimination result is not a reproduction of full SimCLR.